In [18]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
def load_data(path):
    return pd.read_csv(path)

def check_data(df):
    print("Shape:",df.shape)
    print("Missing values:",df.isnull().sum())
    print("Duplicate rows:",df.duplicated().sum())
    print("numeric cols:",df.select_dtypes(include="number").shape[1],"text cols:",df.select_dtypes(exclude="number").shape[1])

def visualize_data(df):
    # 1. Price histograms
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.hist(df["price"], bins=20, edgecolor="black")
    plt.title("Price Distribution")
    plt.xlabel("Price")
    plt.subplot(1, 2, 2)
    plt.hist(np.log1p(df["price"]), bins=20, edgecolor="black")
    plt.title("Log Price Distribution")
    plt.xlabel("Log(1 + Price)")
    plt.tight_layout()
    plt.show()

    # 2. Scatter plots
    cols = ["enginesize", "horsepower", "curbweight", "citympg"]
    for col in cols:
        plt.figure(figsize=(5, 4))
        plt.scatter(df[col], df["price"])
        plt.xlabel(col)
        plt.ylabel("Price")
        plt.title(col + " vs Price")
        plt.show()

    # 3. Correlation bar chart
    corr = df.select_dtypes(include="number").corr()["price"]
    corr = corr.drop("price").sort_values()
    corr.plot(kind="barh", figsize=(8, 6))
    plt.title("Correlation with Price")
    plt.xlabel("Correlation")
    plt.tight_layout()
    plt.show()

    # 4. Boxplots by category
    cols = ["carbody", "fueltype", "aspiration"]
    for col in cols:
        df.boxplot(column="price", by=col, figsize=(6, 4))
        plt.title("Price by " + col)
        plt.suptitle("")
        plt.xlabel(col)
        plt.ylabel("Price")
        plt.show()

def iqr_outliers(df, col):
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    return int(mask.sum())

def encode_features(raw):
    df = raw.drop(columns=["car_ID"]).copy()
    df["brand"] = df["CarName"].str.split().str[0].str.lower()
    df["brand"] = df["brand"].replace({
        "maxda": "mazda",
        "porcshce": "porsche",
        "toyouta": "toyota",
        "vokswagen": "volkswagen",
        "vw": "volkswagen"})
    df = df.drop(columns=["CarName"])

    # encoding
    df["fueltype"] = df["fueltype"].map({"gas": 0, "diesel": 1})
    df["aspiration"] = df["aspiration"].map({"std": 0, "turbo": 1})
    df["doornumber"] = df["doornumber"].map({"two": 2, "four": 4})
    df["enginelocation"] = df["enginelocation"].map({"front": 0, "rear": 1})

    # ordinal column
    df["cylindernumber"] = df["cylindernumber"].map({
        "two": 2, "three": 3, "four": 4, "five": 5,
        "six": 6, "eight": 8, "twelve": 12})

    # one-hot encode the remaining text columns
    cats = ["carbody", "drivewheel", "enginetype", "fuelsystem", "brand"]
    return pd.get_dummies(df, columns=cats, drop_first=True, dtype=float)
def split_xy(df, seed=42):
    X = df.drop(columns="price").to_numpy(dtype=float)
    y = df["price"].to_numpy(dtype=float).reshape(-1, 1)
    np.random.seed(seed)
    idx = np.random.permutation(len(X))
    n_test = int(0.2 * len(X))
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def standardize(X_train, X_test):#zscorenormalisation
    mu, sd = X_train.mean(axis=0), X_train.std(axis=0)
    sd[sd == 0] = 1
    return (X_train - mu) / sd, (X_test - mu) / sd

def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

def predict(X, w):
    return X @ w

def compute_loss(X, y, w):
    error = predict(X, w) - y
    return (error ** 2).mean() / 2

def gradient_descent(X, y, lr=0.1, max_epochs=1000, tol=1e-6):
    w = np.zeros((X.shape[1], 1))     # w[0] is the intercept w0, w[1:] are the feature weights
    losses = []
    for epoch in range(max_epochs):
        error = predict(X, w) - y
        loss = (error ** 2).mean() / 2
        losses.append(loss)
        if epoch > 0 and abs(losses[-2] - loss) / losses[-2] < tol:
            break

        grad = X.T @ error / len(y)
        w = w - lr * grad
    return w, losses



plt.figure(figsize=(8, 5))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss (MSE / 2)")
plt.title("Loss vs Epoch")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
def mae(y, y_hat):
    return np.abs(y - y_hat).mean()

def rmse(y, y_hat):
    return np.sqrt(((y - y_hat) ** 2).mean())

def r2_score(y, y_hat):
    ss_res = ((y - y_hat) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot
raw = load_data("CarPrice_Assignment.csv")
df = encode_features(raw)

X_train, X_test, y_train, y_test = split_xy(df)
X_train, X_test = standardize(X_train, X_test)
X_train, X_test = add_bias(X_train), add_bias(X_test)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
# expect (164, 60) (41, 60) (164, 1) (41, 1)

w, losses = gradient_descent(X_train, y_train, lr=0.1, max_epochs=1000, tol=1e-6)


pred_test = predict(X_test, w)
print("w0 (intercept):", w[0, 0])
print("MAE :", mae(y_test, pred_test))
print("RMSE:", rmse(y_test, pred_test))
print("R2  :", r2_score(y_test, pred_test))    

(164, 60) (41, 60) (164, 1) (41, 1)
w0 (intercept): 13223.414634146358
MAE : 1938.843258046867
RMSE: 2935.609705592847
R2  : 0.8908365162105448
